In [1]:
import numpy as np
import pickle
import pandas as pd
import ast
import sys
sys.path.append('../../src/')
from utilities import print_exams

In [2]:
def pair_representation(serumHA, virusHA, type, mutate_matrix):
    serumChar = [char for char in serumHA]
    virusChar = [char for char in virusHA]

    if len(serumChar) != len(virusChar):
        return [np.nan for _ in range(len(serumChar))]
    
    return diff_calculation(serumChar, virusChar, type=type, mutate_matrix=mutate_matrix)

def diff_calculation(SChar, VChar, type, mutate_matrix):
    difference = []
    for i in range(len(SChar)):
        if type == 'one-hot':
            diff = 0 if SChar[i] == VChar[i] else 1
        if type == 'mut-mat':
            if any(item not in mutate_matrix.columns for item in SChar[i] + VChar[i]):
                diff = 0
            else:
                mut_score = mutate_matrix.loc[VChar[i], VChar[i]]
                ori_score = mutate_matrix.loc[SChar[i], SChar[i]]
                cross_score = mutate_matrix.loc[SChar[i], VChar[i]]
                diff = mut_score + ori_score + cross_score
        difference.append(diff)
    return difference

def predict_Adaboost_titer(test_df, meta_features, model_path=None):
    # 读取模型和编码器
    with open(model_path, 'rb') as file:
        H1N1_result = pickle.load(file)

    ohe = H1N1_result['Encoder']
    model = H1N1_result['model']

    # 元特征 one-hot
    meta = test_df[meta_features].fillna('None').astype('str')
    meta = ohe.transform(meta).toarray()

    # 提取序列差异特征
    seq_diff = np.array(test_df['seq_diff_ohe'].tolist())[:, 16:345]

    # 拼接数据
    data_array = np.hstack((seq_diff, meta))

    # 去掉含 NaN 的行
    nan_mask = pd.isna(data_array).any(axis=1)
    X_test = data_array[~nan_mask]

    # 标签和预测
    label = test_df['label'].loc[~nan_mask]
    prediction = model.predict(X_test).tolist()

    return label, prediction


In [3]:
## meta feature for model training
meta_features = ['virusName',   # virus avidity (based on both name and passage)
                'serumName',   # antiserum potency (based on both name and passage)
                'virusPassCat',   # virus passage category
                'serumPassCat']   # serum passage category


In [41]:
season = '2025SH'

test_df = pd.read_csv(f'../../data/reverse_test/processed/test_{season}/test.csv', index_col=False)
test_df['seq_diff_ohe'] = test_df.apply(
        lambda row: pair_representation(row['serumHA'], row['virusHA'], type='one-hot', mutate_matrix=None),
        axis=1
    )

In [42]:
H1_path = f'./{season}/Adaboost/Adaboost_H1N1_titer.pkl'
H3_path = f'./{season}/Adaboost/Adaboost_H3N2_titer.pkl'

H1N1_label, H1N1_prediction = predict_Adaboost_titer(test_df[test_df['Type'] == 'H1N1'], meta_features, H1_path)
H3N2_label, H3N2_prediction = predict_Adaboost_titer(test_df[test_df['Type'] == 'H3N2'], meta_features, H3_path)

test_df.loc[test_df['Type'] == 'H1N1', 'pred_with_name'] = H1N1_prediction
test_df.loc[test_df['Type'] == 'H3N2', 'pred_with_name'] = H3N2_prediction
result = print_exams(test_df['label'], test_df['pred_with_name'])

MAE: 1.00115
MSE: 2.05605
pearson correlation: 0.71333
spearman correlation: 0.67021
R2_score: 0.45021


In [43]:
temp_df = test_df.copy()
temp_df.loc[:, 'serumName'] = ''
temp_df.loc[:, 'virusName'] = ''
H1N1_label, H1N1_prediction = predict_Adaboost_titer(temp_df[temp_df['Type'] == 'H1N1'], meta_features, H1_path)
H3N2_label, H3N2_prediction = predict_Adaboost_titer(temp_df[temp_df['Type'] == 'H3N2'], meta_features, H3_path)

test_df.loc[test_df['Type'] == 'H1N1', 'pred_without_name'] = H1N1_prediction
test_df.loc[test_df['Type'] == 'H3N2', 'pred_without_name'] = H3N2_prediction
result = print_exams(test_df['label'], test_df['pred_without_name'])

MAE: 1.08828
MSE: 2.28902
pearson correlation: 0.68020
spearman correlation: 0.65127
R2_score: 0.38791


In [46]:
print(test_df.shape[0], test_df[test_df['Type'] == 'H1N1'].shape[0], test_df[test_df['Type'] == 'H3N2'].shape[0])

4592 2067 2525


In [47]:
save_path = f'/home/chenyh/workspace/fluProfiler/experiments/reverse_tests/test_result/{season}_Adaboost.csv'
test_df.to_csv(save_path, index=False)